# Session 2: プログラムで自律飛行
# Session 2: Autonomous Flight with Programming

## 目標 / Objective

矩形パスを自律飛行し、ログを取得して軌跡をプロットする。

Fly a rectangular path autonomously, capture logs, and plot the trajectory.

## この Notebook で学ぶこと / What You'll Learn

| トピック | 説明 |
|---------|------|
| 自律飛行 | 複数の移動コマンドを連結 |
| ログ解析 | CSV ログを pandas で読み込み |
| 軌跡プロット | XY 平面の飛行経路を可視化 |
| 精度評価 | 理想経路と実際の経路を比較 |

## 1. 環境セットアップ / Setup

In [ ]:
from stampfly_edu import (
    connect_or_simulate,
    load_flight_log,
    load_sample_data,
    plot_trajectory,
    compare_logs,
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Ready! / 準備完了！")

## 2. 理論: 経路計画 / Theory: Path Planning

矩形パス飛行は以下のウェイポイントで構成されます：

```
  (0,1) ←──── (1,1)
    │              ↑
    ↓              │
  (0,0) ────→ (1,0)
  START
```

各辺を飛行する際、制御系は以下を同時に処理します：

1. **位置制御**: 目標座標への移動
2. **高度制御**: 一定高度の維持
3. **姿勢制御**: 水平の維持

### 誤差の原因 / Sources of Error

| 原因 | 影響 |
|------|------|
| センサノイズ | 位置推定の揺らぎ |
| 風外乱 | 経路の偏差 |
| 制御応答遅れ | コーナーでの overshoot |
| オプティカルフロー | 速度推定精度 |

## 3. 矩形パス飛行 / Square Path Flight

### 実機飛行（またはシミュレータ）

In [ ]:
# Connect and fly a square path
# 接続して矩形パスを飛行
drone = connect_or_simulate()

drone.takeoff()

# Fly 1m square (100cm per side)
# 1m 正方形を飛行
SIDE_CM = 100

print("Side 1: Forward / 辺1: 前進")
drone.move_forward(SIDE_CM)

print("Side 2: Left / 辺2: 左")
drone.move_left(SIDE_CM)

print("Side 3: Back / 辺3: 後退")
drone.move_back(SIDE_CM)

print("Side 4: Right / 辺4: 右")
drone.move_right(SIDE_CM)

drone.land()
print("Flight complete! / 飛行完了！")

## 4. ログ解析 / Log Analysis

飛行データを読み込んで分析します。
実機データがない場合はサンプルデータを使用します。

In [ ]:
# Load sample square path data
# サンプル矩形パスデータを読み込む
log = load_sample_data("square_path")

print(f"Data shape: {log.shape}")
print(f"Columns: {list(log.columns)}")
print(f"Duration: {log['time'].iloc[-1]:.1f} s")
log.head()

In [ ]:
# Basic statistics
# 基本統計量
log[["x", "y", "z"]].describe()

## 5. 軌跡プロット / Trajectory Plot

In [ ]:
# Plot XY trajectory
# XY 軌跡をプロット
ax = plot_trajectory(log, title="Square Path Flight / 矩形パス飛行")

# Overlay ideal path
# 理想経路を重ねて表示
ideal_x = [0, 1, 1, 0, 0]
ideal_y = [0, 0, 1, 1, 0]
ax.plot(ideal_x, ideal_y, "g--", linewidth=2, alpha=0.5, label="Ideal / 理想")
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Plot 3D trajectory
# 3D 軌跡をプロット
fig = plt.figure(figsize=(10, 8))
ax3d = fig.add_subplot(111, projection="3d")

ax3d.plot(log["x"], log["y"], log["z"], "b-", linewidth=1.5)
ax3d.scatter(log["x"].iloc[0], log["y"].iloc[0], log["z"].iloc[0],
             color="green", s=100, label="Start / 開始")
ax3d.scatter(log["x"].iloc[-1], log["y"].iloc[-1], log["z"].iloc[-1],
             color="red", s=100, label="End / 終了")

ax3d.set_xlabel("X (m)")
ax3d.set_ylabel("Y (m)")
ax3d.set_zlabel("Z (m)")
ax3d.set_title("3D Flight Path / 3D 飛行経路")
ax3d.legend()

plt.tight_layout()
plt.show()

## 6. 精度評価 / Accuracy Evaluation

理想経路と実際の経路の距離（RMSE）を計算して定量評価します。

In [ ]:
def compute_path_error(log, ideal_waypoints):
    """Compute minimum distance from each point to ideal path segments.

    各点から理想経路セグメントへの最小距離を計算する。
    """
    from scipy.spatial.distance import cdist

    errors = []
    for _, row in log.iterrows():
        point = np.array([row["x"], row["y"]])
        min_dist = float("inf")

        for i in range(len(ideal_waypoints) - 1):
            a = np.array(ideal_waypoints[i])
            b = np.array(ideal_waypoints[i + 1])

            # Project point onto line segment
            # 点を線分に射影
            ab = b - a
            ap = point - a
            t = np.clip(np.dot(ap, ab) / (np.dot(ab, ab) + 1e-10), 0, 1)
            closest = a + t * ab
            dist = np.linalg.norm(point - closest)
            min_dist = min(min_dist, dist)

        errors.append(min_dist)

    return np.array(errors)

# Ideal waypoints
# 理想ウェイポイント
waypoints = [(0, 0), (1, 0), (1, 1), (0, 1), (0, 0)]

# Only evaluate path segments (exclude takeoff/landing)
# 経路セグメントのみ評価（離着陸を除外）
path_log = log[(log["z"] > 0.1)]

if len(path_log) > 0:
    errors = compute_path_error(path_log, waypoints)
    
    print(f"Path tracking error / 経路追従誤差:")
    print(f"  Mean / 平均: {np.mean(errors)*100:.1f} cm")
    print(f"  Max / 最大:  {np.max(errors)*100:.1f} cm")
    print(f"  RMSE:        {np.sqrt(np.mean(errors**2))*100:.1f} cm")
    
    # Plot error over time
    # 時間に対する誤差をプロット
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(path_log["time"], errors * 100, "r-")
    ax.set_xlabel("Time (s) / 時間")
    ax.set_ylabel("Path Error (cm) / 経路誤差")
    ax.set_title("Path Tracking Error / 経路追従誤差")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 7. 時系列分析 / Time Series Analysis

X, Y, Z 座標の時間変化を個別に見てみましょう。

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

for ax, col, label in zip(axes, ["x", "y", "z"],
                           ["X (m)", "Y (m)", "Z (m)"]):
    ax.plot(log["time"], log[col], "b-", linewidth=1.0)
    ax.set_ylabel(label)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Time (s) / 時間")
fig.suptitle("Position vs Time / 位置の時間変化", fontsize=14)
fig.tight_layout()
plt.show()

## 8. チャレンジ課題 / Challenges

### 課題 1: 三角形パス
正三角形（一辺 80cm）のパスを飛行するプログラムを書いてください。
ヒント: `rotate_clockwise()` を使って方向転換

### 課題 2: 速度と精度のトレードオフ
移動距離を変えて（50cm, 100cm, 150cm）矩形パスを飛び、精度を比較してください。

### 課題 3: 繰り返し精度
同じ矩形パスを3回繰り返し飛行し、再現性を評価してください。

---

### Challenge 1: Triangle Path
Write a program to fly an equilateral triangle (80cm per side).
Hint: Use `rotate_clockwise()` for turns.

### Challenge 2: Speed vs Accuracy Trade-off
Fly square paths with different sizes (50cm, 100cm, 150cm) and compare accuracy.

### Challenge 3: Repeatability
Fly the same square path 3 times and evaluate reproducibility.

In [ ]:
# Your code here / ここにコードを書いてください
# Challenge 1: Triangle path
# 課題 1: 三角形パス


In [ ]:
# Clean up
# 後片付け
drone.end()
print("Session complete! / セッション完了！")